In [1]:
# 1. Verify GPU runtime
!nvidia-smi

Thu Aug 27 03:11:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pwd
!ls


/content
drive  sample_data


In [4]:
# Find repo on Colab — search Google Drive first, then /content
from pathlib import Path

MARKER = Path("data-generation") / "generate_story_chunking_dataset.py"

def find_repo_root() -> Path | None:
    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob("generate_story_chunking_dataset.py"):
            if path.parent.name == "data-generation":
                return path.parent.parent
    return None

REPO_ROOT = find_repo_root()

if REPO_ROOT:
    print("Found repo:", REPO_ROOT)
else:
    print("NOT FOUND on Colab yet.")
    print("Your project is on local OneDrive — it is NOT auto-synced to Google Drive.")
    print("Run the next cell to upload a zip of the 'Model Training' folder.")

FileNotFoundError: Could not find Model Training repo root. Ensure the workspace is synced to Colab or cd to the repo.

In [ ]:
# ONLY run if cell above printed NOT FOUND
# On your PC: right-click "Model Training" folder -> Send to -> Compressed (zipped) folder
# Then run this cell and select that zip file.
from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content")

REPO_ROOT = None
for path in Path("/content").rglob("generate_story_chunking_dataset.py"):
    if path.parent.name == "data-generation":
        REPO_ROOT = path.parent.parent
        break

if REPO_ROOT is None:
    raise FileNotFoundError("Could not find data-generation/ after unzip. Check zip structure.")

print("Repo ready at:", REPO_ROOT)

In [ ]:
# Install dependencies
import importlib.util
import os

if REPO_ROOT is None:
    raise RuntimeError("REPO_ROOT not set — run find or upload cell first.")

os.chdir(REPO_ROOT)

if importlib.util.find_spec("transformers") is None:
    get_ipython().system('pip install -q -r data-generation/requirements.txt')
else:
    print("Dependencies already installed.")

In [ ]:
# 4. Check CUDA + Hugging Face token
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    print("WARNING: HF_TOKEN not set. Run: huggingface-cli login")
    print("And accept the Meta Llama 3.1 license at huggingface.co")
else:
    print("HF_TOKEN: set")

In [ ]:
# Smoke test — generate ONE V2 example on GPU
import os
os.chdir(REPO_ROOT)

get_ipython().system('python data-generation/generate_story_chunking_dataset.py --smoke-test --provider transformers --model meta-llama/Meta-Llama-3.1-8B-Instruct --output story-chunking/data/training/v2/synthetic_story_chunking_v2.jsonl')

In [ ]:
# Full 20-example V2 run (only after smoke test passes)
import os
os.chdir(REPO_ROOT)

get_ipython().system('python data-generation/generate_story_chunking_dataset.py --count 20 --provider transformers --model meta-llama/Meta-Llama-3.1-8B-Instruct --output story-chunking/data/training/v2/synthetic_story_chunking_v2.jsonl')